# 02 — XAI: wichtige Segmente markieren und das Modell schrittweise aufbauen

Zuerst werden wichtige Heiz-/Abkühlsegmente auf dem echten Zyklus farbig markiert. Danach wird aus Top 1, Top 2, … schrittweise ein reduziertes Modell gebaut.


In [ ]:
from pathlib import Path
import sys,numpy as np,matplotlib.pyplot as plt
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'Networks'/'TCOCNNv3.py').exists())
sys.path.insert(0,str(ROOT/'Networks'))
sys.path.insert(0,str(ROOT/'Evaluation Seminar'/'Day_03'))
sys.path.insert(0,str(ROOT/'Evaluation Seminar'/'Day_04'))
from day3_utils import show_results
from day4_utils import load_checkpoint,transfer_subset,fit_network,predict_ppb
from transfer_workflow import metrics,plot_predictions,copy_state,save_json,pool_domains
from calibration_transfer_sensor0 import load_domains,scale_domain
OUTPUT=ROOT/'artifacts'/'seminar_day4_global6_sensor0'
model,scaler,metadata=load_checkpoint(OUTPUT/'source_global6')
raw=load_domains(metadata['gas']); SENSORS=list(raw); TARGET=metadata['target']
domain=scale_domain(raw[TARGET],scaler); subset=transfer_subset(domain,40,seed=metadata['seed'])
reference=subset['train']['X_z'].mean(axis=0,keepdims=True); source_state=copy_state(model)
GAS_NAME=metadata['gas']; UNIT='ppb'
N_VAL_UGMS=len(np.unique(domain['val']['groups'])); N_TEST_UGMS=len(np.unique(domain['test']['groups']))
print('Erklärtes Modell: Global-6 mit Subsensor 0 jedes Geräts')
print('Ranking:',len(np.unique(subset['train']['groups'])),'UGMs;',len(subset['train']['y']),'Zyklen')
print('Validation:',len(np.unique(domain['val']['groups'])),'UGMs; Test:',N_TEST_UGMS,'UGMs')


## 1. Segmente verdecken, bewerten und direkt im Zyklus markieren

Je größer die Vorhersageänderung beim Ersetzen eines Segments, desto stärker nutzt das Modell dieses Segment. Das ist Modellabhängigkeit, kein chemischer Kausalitätsbeweis.


In [ ]:
X=subset['train']['X_z']; original=predict_ppb(model,{'X_z':X},scaler)
HIGH_SAMPLES=50; LOW_SAMPLES=70; PAIR_SAMPLES=HIGH_SAMPLES+LOW_SAMPLES
if X.shape[2] % PAIR_SAMPLES:
    raise ValueError(f'Signal length {X.shape[2]} is not divisible by the configured high/low pair length {PAIR_SAMPLES}.')
N_PAIRS=X.shape[2]//PAIR_SAMPLES
effects={kind:[] for kind in ['high','low','pair']}
for pair in range(N_PAIRS):
    start=pair*PAIR_SAMPLES; split=start+HIGH_SAMPLES; stop=(pair+1)*PAIR_SAMPLES
    for kind,a,b in [('high',start,split),('low',split,stop),('pair',start,stop)]:
        masked=X.copy(); masked[:,:,a:b]=reference[:,:,a:b]
        effects[kind].append(original-predict_ppb(model,{'X_z':masked},scaler))
effects={kind:np.stack(value,axis=1) for kind,value in effects.items()}
groups=subset['train']['groups']; unique=np.unique(groups)
importance={kind:np.mean([np.abs(value[groups==group]).mean(axis=0) for group in unique],axis=0) for kind,value in effects.items()}
ranking=np.argsort(-importance['pair'],kind='stable')
show_results([{'rank':rank+1,'pair':int(pair+1),'high_samples':HIGH_SAMPLES,'low_samples':LOW_SAMPLES,
               'sample_start':int(pair*PAIR_SAMPLES),'sample_stop_exclusive':int((pair+1)*PAIR_SAMPLES),
               'mean_abs_occlusion_ppb':importance['pair'][pair]} for rank,pair in enumerate(ranking)])
fig,axes=plt.subplots(1,3,figsize=(20,5))
for kind in ['high','low','pair']: axes[0].plot(np.arange(1,N_PAIRS+1),importance[kind],marker='o',label=kind)
axes[0].set(xlabel='Original high/low pair',ylabel='Mean absolute prediction change [ppb]',title=f'{GAS_NAME}: {len(np.unique(subset["train"]["groups"]))} training UGMs'); axes[0].legend()
limit=max(float(np.abs(effects['pair']).max()),1e-6)
im=axes[1].imshow(effects['pair'],aspect='auto',cmap='coolwarm',vmin=-limit,vmax=limit)
axes[1].set(xlabel='Segmentindex (nullbasiert)',ylabel='Adaptionszyklus',title='Vorzeichenbehaftete Occlusion [ppb]'); fig.colorbar(im,ax=axes[1])
signal=subset['train']['X_z'][:,0,:,0].mean(axis=0)
axes[2].plot(signal,color='black',lw=1)
colors=plt.cm.viridis(np.linspace(.15,.9,min(5,N_PAIRS)))
for rank,pair in enumerate(ranking[:5]):
    start=pair*PAIR_SAMPLES; stop=(pair+1)*PAIR_SAMPLES
    axes[2].axvspan(start,stop,color=colors[rank],alpha=.35,label=f'Rang {rank+1}: Segment {pair+1}')
axes[2].set(xlabel='Zeitindex',ylabel='mittleres Signal z',title='Die fünf wichtigsten Segmente')
axes[2].legend(fontsize=8)
for ax in axes: ax.grid(True,alpha=.3)
plt.tight_layout(); plt.show()


## 2. Aus Top 1 bis Top 12 schrittweise ein Modell bauen

Für jedes k werden nur die k wichtigsten vollständigen Heiz-/Abkühlpaare behalten. Jedes Modell startet vom selben Global‑6-Checkpoint. Validation wählt k; Test bleibt bis danach geschlossen.


In [ ]:
rows=[]; val_predictions={}; masked_val={}; reduced_models={}
for k in range(1,N_PAIRS+1):
    kept=np.sort(ranking[:k]); positions=np.concatenate([np.arange(pair*PAIR_SAMPLES,(pair+1)*PAIR_SAMPLES) for pair in kept])
    reduced={name:{**split,'X_z':split['X_z'][:,:,positions,:]} for name,split in subset.items()}
    reduced_model,_,info=fit_network(reduced,metadata['params'],epochs=40,
                                    initial=source_state,seed=metadata['seed'],restore_best=False,
                                    use_validation=True,freeze_batchnorm=True,lr_schedule='constant',verbose=False)
    reduced_models[k]=(reduced_model,positions)
    val_predictions[k]=predict_ppb(reduced_model,reduced['val'],scaler)
    masked=np.broadcast_to(reference,domain['val']['X_z'].shape).copy()
    masked[:,:,positions,:]=domain['val']['X_z'][:,:,positions,:]
    masked_val[k]=predict_ppb(model,{'X_z':masked},scaler)
    score=metrics(domain['val']['y'],val_predictions[k],domain['val']['groups'])
    mask_score=metrics(domain['val']['y'],masked_val[k],domain['val']['groups'])
    rows.append({'k':k,'kept_pairs':','.join(str(int(p+1)) for p in kept),'samples':PAIR_SAMPLES*k,'nominal_seconds':PAIR_SAMPLES*k/10.,
                 'retrained_val_UGM_RMSE_ppb':score['UGM_RMSE_ppb'],'masked_val_UGM_RMSE_ppb':mask_score['UGM_RMSE_ppb']})
show_results(rows)
best_k=min(rows,key=lambda row:row['retrained_val_UGM_RMSE_ppb'])['k']
print('Frozen validation-selected k:',best_k)
print('Test split has not been predicted yet.')
fig,ax=plt.subplots(figsize=(11,5))
ax.plot([r['nominal_seconds'] for r in rows],[r['retrained_val_UGM_RMSE_ppb'] for r in rows],marker='o',label='Reduced-input adaptation')
ax.plot([r['nominal_seconds'] for r in rows],[r['masked_val_UGM_RMSE_ppb'] for r in rows],marker='s',label='Masking, no retraining')
ax.axvline(PAIR_SAMPLES*best_k/10.,color='black',linestyle=':',label='Validation optimum')
ax.set(xlabel='Retained samples / 10 Hz [s]',ylabel=f'{GAS_NAME} validation UGM RMSE [{UNIT}]',title=f'Top 1 to Top {N_PAIRS}; {N_VAL_UGMS} validation UGMs'); ax.legend()
ax.grid(True,which='both',alpha=.3)
plt.tight_layout(); plt.show()


## 3. Eingefrorene Auswahl auf dem vollständigen Testsplit

Erst jetzt werden alle Testzyklen vorhergesagt. Ein zufällig besseres Test-k ersetzt nicht nachträglich die Validierungsauswahl.


In [ ]:
final=[]; test_predictions={}
print('Opening XAI test only after k selection has been frozen.')
for k in range(1,N_PAIRS+1):
    reduced_model,positions=reduced_models[k]
    reduced_test={**domain['test'],'X_z':domain['test']['X_z'][:,:,positions,:]}
    test_predictions[k]=predict_ppb(reduced_model,reduced_test,scaler)
    score=metrics(domain['test']['y'],test_predictions[k],domain['test']['groups'])
    final.append({'k':k,'nominal_seconds':PAIR_SAMPLES*k/10.,'chosen_on_validation':k==best_k,**score})
show_results(final)
fig,ax=plt.subplots(figsize=(11,5))
ax.plot([PAIR_SAMPLES*k/10. for k in range(1,N_PAIRS+1)],[r['UGM_RMSE_ppb'] for r in final],marker='o',label='Test UGM RMSE')
ax.axvline(PAIR_SAMPLES*best_k/10.,color='black',linestyle=':',label='Validation-selected k')
ax.set(xlabel='Retained samples / 10 Hz [s]',ylabel=f'{GAS_NAME} test UGM RMSE [{UNIT}]',title=f'{N_TEST_UGMS} unseen test UGMs; selection already frozen'); ax.legend()
ax.grid(True,which='both',alpha=.3)
plt.tight_layout(); plt.show()
selected=list(dict.fromkeys([1,2,best_k,N_PAIRS]))
plot_predictions(domain['test'],{f'Top {k} pairs':test_predictions[k] for k in selected},f'{GAS_NAME}: reduced-input test comparison')
save_json(output/'occlusion_cycle_buildup.json',{'ranking_zero_based':ranking.tolist(),'validation':rows,'selected_k':best_k,'test':final})
